# **Day 13: RSA & Lattice-Based Cryptography**
---

### **Description**
In this notebook, we get hands-on with the two ends of the public-key story. First we build **RSA** from scratch. This includes generating keys, encrypting, and decrypting. We will then watch how its security (the difficulty of factoring) explodes as we add bits, until it becomes hopeless for our computer.

Then we move to **lattice-based post-quantum cryptography**: we build a 2D lattice, see how the *same* lattice can be described by a "good" or a "bad" basis, hide a message as a point near the lattice (the CVP), and discover that only the good (secret) basis can decrypt it.

<br>

### **Structure**
**Part 1**: [RSA & the Factoring Problem](#p1)

**Part 2**: [Lattice-Based Post-Quantum Cryptography](#p2)

<br>

### **Resources**
* Slides: *RSA & Lattice-Based Encryption* (Cryptography Lab)
* NIST post-quantum standards: FIPS 203 (ML-KEM), FIPS 204 (ML-DSA)

<br>

**Before starting, run the code cell below to import all necessary libraries and helper functions.**

In [ ]:
# @title Run this cell to set up the lab { display-mode: "form" }
import warnings
warnings.filterwarnings('ignore')

import math, time, random
import numpy as np
import matplotlib.pyplot as plt

try:
    from sympy import randprime, gcd
except ImportError:
    print('installing sympy...')
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'sympy', '--quiet'])
    from sympy import randprime, gcd

np.set_printoptions(suppress=True)


# ----------------------------------------------------------------------
# RSA HELPERS
# ----------------------------------------------------------------------
def generate_rsa_keypair(bits, seed=None):
    """
    Generate an RSA key pair whose modulus n is exactly `bits` bits long.

    Returns a dictionary with p, q, n, phi, e, d.
      Public key  = (e, n)   ->  used to ENCRYPT:  c = m**e mod n
      Private key = (d, n)   ->  used to DECRYPT:  m = c**d mod n
    """
    if seed is not None:
        random.seed(seed)
    half = bits // 2
    for _ in range(5000):
        p = randprime(2 ** (half - 1), 2 ** half)
        q = randprime(2 ** (half - 1), 2 ** half)
        if p == q:
            continue
        n = p * q
        if n.bit_length() != bits:
            continue
        phi = (p - 1) * (q - 1)
        e = 3
        while gcd(e, phi) != 1:
            e += 2
        d = pow(e, -1, phi)          # modular inverse of e mod phi
        return {'p': p, 'q': q, 'n': n, 'phi': int(phi), 'e': int(e), 'd': int(d)}
    raise RuntimeError('key generation failed for %d bits' % bits)


def factor_by_trial_division(n, time_budget=None):
    """
    Try to factor n by trial division (the most basic attack on RSA).
    Returns (factor, elapsed_seconds, finished).
      - finished == True  -> we found a factor (or proved none <= sqrt(n))
      - finished == False -> we ran out of the time_budget first
    """
    start = time.perf_counter()
    if n % 2 == 0:
        return 2, time.perf_counter() - start, True
    limit = math.isqrt(n)
    i = 3
    while i <= limit:
        if n % i == 0:
            return i, time.perf_counter() - start, True
        i += 2
        if time_budget is not None and (i & 0x3FFFF) == 1:
            if time.perf_counter() - start > time_budget:
                return None, time.perf_counter() - start, False
    return None, time.perf_counter() - start, True


# ----------------------------------------------------------------------
# LATTICE HELPERS
# ----------------------------------------------------------------------
def angle_between(u, v):
    """Angle (in degrees) between two 2D vectors."""
    c = np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))
    return math.degrees(math.acos(max(-1.0, min(1.0, c))))


def lattice_points(B, rng=6):
    """All points a*b1 + c*b2 for integers a,c in [-rng, rng]. B has basis vectors as ROWS."""
    pts = []
    for a in range(-rng, rng + 1):
        for c in range(-rng, rng + 1):
            pts.append(a * B[0] + c * B[1])
    return np.array(pts)


def babai_round(t, B):
    """
    Babai's rounding algorithm: find the lattice point closest to target t,
    using basis B (rows = basis vectors).
    Solve t = coords @ B, round coords to integers, map back to a lattice point.
    """
    coords = np.linalg.solve(B.T, t)
    rounded = np.round(coords)
    return rounded @ B, rounded.astype(int)


print('Setup complete. Helper functions ready:')
print('  RSA     -> generate_rsa_keypair(bits), factor_by_trial_division(n)')
print('  Lattice -> angle_between(u,v), lattice_points(B), babai_round(t, B)')

Setup complete. Helper functions ready:
  RSA     -> generate_rsa_keypair(bits), factor_by_trial_division(n)
  Lattice -> angle_between(u,v), lattice_points(B), babai_round(t, B)


<a name="p1"></a>

---
## **Part 1: RSA & the Factoring Problem**
---

RSA is the classic public-key cryptosystem. Its security rests on one idea: **multiplying two primes is easy, but factoring their product back into those primes is hard.** In this part we build a tiny RSA system, use it, and then watch the "hard" direction get exponentially harder as the key grows.

#### **Problem #1.1**

Before we let the computer do it, let's build an RSA key pair **entirely by hand** using two small primes, so you can see exactly where the public and private keys come from.

Follow the steps:

1. Choose two prime numbers: `p = 3` and `q = 11`.
2. Compute the **modulus** `n = p * q`.
3. Compute **Euler's totient** `phi = (p - 1) * (q - 1)`.
4. Choose a **public exponent** `e = 3` (it must satisfy `gcd(e, phi) == 1`).
5. Find the **private exponent** `d`: the number such that `(e * d) % phi == 1` (the *modular inverse* of `e`). Find it by testing `d = 1, 2, 3, ...` until it works.
6. Assemble the **public key** `(e, n)` and the **private key** `(d, n)`.

In [ ]:
# Step 1: choose two small primes
p = 3
q = 11

# Step 2: the modulus
n = # COMPLETE THIS CODE

# Step 3: Euler's totient
phi = # COMPLETE THIS CODE

# Step 4: public exponent (must have gcd(e, phi) == 1)
e = 3

# Step 5: find the private exponent d such that (e * d) % phi == 1
d = None
for candidate in range(1, phi):
    if # COMPLETE THIS CODE :
        d = candidate
        break

# Step 6: assemble and print the keys
print('n =', n, '   phi =', phi)
print('PUBLIC KEY  (e, n) =', (e, n))
print('PRIVATE KEY (d, n) =', (d, n))

#### **Problem #1.2**

Now use the keys you just built to send a secret message.

1. Pick a message `m` — any number smaller than `n = 33`. Try `m = 9`.
2. **Encrypt** with the *public* key: `c = m**e mod n`.
3. **Decrypt** with the *private* key: `m = c**d mod n`.
4. Confirm you recover your original message.

Use Python's `pow(base, exponent, modulus)` — it does fast modular exponentiation for you.

In [ ]:
m = 9                       # your secret message (must be < n = 33)

# Encrypt with the PUBLIC key (e, n)
c = # COMPLETE THIS CODE     # hint: pow(m, e, n)

# Decrypt with the PRIVATE key (d, n)
recovered = # COMPLETE THIS CODE

print('message   m =', m)
print('encrypted c =', c)
print('decrypted   =', recovered)

#### **Problem #1.3**

Building keys by hand is fine for tiny primes, but real RSA uses enormous ones. The helper `generate_rsa_keypair(bits)` runs the **exact same six steps** automatically. Use it to make a **10-bit** key, then encrypt and decrypt a message.

1. Call `generate_rsa_keypair(10, seed=13)` and read off `p, q, n, phi, e, d`.
2. Encrypt `m = 42` with the public key and decrypt it back with the private key.

In [ ]:
# Step 1: Generate a 10-bit RSA key pair
keys = generate_rsa_keypair(# COMPLETE THIS CODE

# Step 2: Pull out the pieces
p, q, n, phi, e, d = # COMPLETE THIS CODE

# Step 3: Encrypt the message m = 42 with the PUBLIC key (e, n)
m = 42
c = # COMPLETE THIS CODE

# Step 4: Decrypt the ciphertext with the PRIVATE key (d, n)
recovered = # COMPLETE THIS CODE

print('recovered message:', recovered)

#### **Problem #1.4**

Anyone can see the public modulus `n`. To break RSA, an attacker must **factor** it back into `p` and `q`. Let's watch how hard that gets as we add bits.

For each key size, generate an RSA modulus, factor it by trial division, and record the time. Then plot **time vs. number of bits** on a log scale. What kind of growth do you see?

In [ ]:
bit_sizes = [10, 16, 20, 24, 28, 32, 36, 40, 44]
times = []

for bits in bit_sizes:
    keys = generate_rsa_keypair(bits, seed=bits)
    n = keys['n']
    factor, elapsed, finished = # COMPLETE THIS CODE  (call factor_by_trial_division)
    times.append(elapsed)
    print(f'{bits:>2}-bit  n = {n:<16}  factored in {elapsed:.5f} s  ({factor} x {n//factor})')

# Plot time vs bits on a log-scale y-axis
# COMPLETE THIS CODE

#### **Problem #1.5**

Trial division checks divisors up to `sqrt(n)`. For a real RSA key (2048 bits), that is about `2^1024` steps. Let's estimate how long that would take.

1. Measure how many trial-division steps *this* computer does per second.
2. For 32, 64, 128, 256, and 2048-bit keys, estimate the number of steps (`~ sqrt(n) = 2^(bits/2)`) and convert to years.
3. Compare the 2048-bit estimate to the age of the universe (~1.38 x 10^10 years).

*Hint: 2048-bit numbers are astronomically large, so work with `log10` of the values instead of the numbers themselves.*

In [ ]:
# 1. Measure this PC's trial-division rate (steps per second)
start = time.perf_counter(); steps = 0; x = 2**44 + 7; i = 3
while time.perf_counter() - start < 0.5:
    for _ in range(200000):
        _ = x % i; i += 2; steps += 1
rate = steps / (time.perf_counter() - start)
print(f'This computer does about {rate:,.0f} trial-division steps per second.\n')

LOG2 = math.log10(2)
LOG_YEAR = math.log10(365.25 * 24 * 3600)
LOG_AGE  = math.log10(1.38e10)

for bits in [32, 64, 128, 256, 2048]:
    # log10 of the number of steps ~ sqrt(n) = 2**(bits/2)
    log_steps = # COMPLETE THIS CODE
    log_years = log_steps - math.log10(rate) - LOG_YEAR
    # COMPLETE THIS CODE  (print the estimate for this key size)

<a name="p2"></a>

---
## **Part 2: Lattice-Based Post-Quantum Cryptography**
---

A quantum computer running **Shor's algorithm** factors large numbers efficiently. We need hard problems that quantum computers *don't* crack. Lattices provide them. A lattice is just a regular grid of points built from a set of **basis vectors**, and the security comes from the **Closest Vector Problem (CVP)**: given a random point, find the nearest lattice point. Easy with a *good* basis; very hard with a *bad* one.

#### **Problem #2.1**

Define a **good basis** for our lattice: two vectors that are short and close to perpendicular. Use

$$b_1 = (3, 1), \qquad b_2 = (1, 3)$$

Store them as the **rows** of a NumPy array `B_good`, then print the basis, its determinant (`np.linalg.det`), and the angle between the two vectors (`angle_between`). The determinant's absolute value is the area of one lattice "cell".

In [ ]:
# Define the two basis vectors as ROWS of a matrix
B_good = np.array(# COMPLETE THIS CODE

print('Good basis (rows are b1, b2):')
print(B_good)
print('determinant =', # COMPLETE THIS CODE
print('angle between b1 and b2 =', # COMPLETE THIS CODE , 'degrees')

#### **Problem #2.2**

Use the two basis vectors to generate the **2D lattice**: every point of the form `a*b1 + c*b2` for integers `a, c`. Use the helper `lattice_points(B_good)` and plot the points. Also draw the two basis vectors as arrows from the origin.

In [ ]:
pts = lattice_points(B_good)

plt.figure(figsize=(6, 6))
plt.scatter(# COMPLETE THIS CODE , s=25, color='#003865')
# Draw the basis vectors as arrows
plt.arrow(0, 0, B_good[0,0], B_good[0,1], # COMPLETE THIS CODE
plt.arrow(0, 0, B_good[1,0], B_good[1,1], # COMPLETE THIS CODE
plt.gca().set_aspect('equal'); plt.grid(alpha=0.3)
plt.xlim(-12, 12); plt.ylim(-12, 12)
plt.title('2D lattice from the good basis')
plt.show()

#### **Problem #2.3**

The **same lattice** can be described by many different bases. A **bad basis** uses long, nearly-parallel vectors but it still generates *exactly the same grid of points*.

Below we build a bad basis by combining the good vectors with a **unimodular** matrix `U` (this guarantees the same lattice). We search for the `U` whose two resulting vectors are only about **10° apart**. Plot the same lattice with the bad basis vectors overlaid, and confirm the determinant (cell area) is unchanged.

In [ ]:
# Search over unimodular matrices U (det = +/-1) for a basis whose
# two vectors are ~10 degrees apart (nearly parallel = a BAD basis)
best = None
for a in range(-9, 10):
  for b in range(-9, 10):
    for cc in range(-9, 10):
      for dd in range(-9, 10):
        if abs(a*dd - b*cc) == 1:              # unimodular -> same lattice
            U = np.array([[a, b], [cc, dd]])
            B_try = U @ B_good
            ang = angle_between(B_try[0], B_try[1])
            if best is None or abs(ang - 10) < best[0]:
                best = (abs(ang - 10), ang, U, B_try)

_, bad_angle, U, B_bad = best
print('Unimodular U =', U.tolist())
print('Bad basis rows =', np.round(B_bad, 2).tolist())
print('angle between bad vectors =', round(bad_angle, 2), 'degrees')
print('det(good) =', round(np.linalg.det(B_good), 2),
      '  det(bad) =', # COMPLETE THIS CODE , '  -> same |area|, same lattice')

# Plot the SAME lattice with the bad basis overlaid
# COMPLETE THIS CODE

#### **Problem #2.4**

Now we **encrypt** a message using the lattice (a simplified GGH scheme):

1. Encode the message as integer coordinates, e.g. `msg = (2, -1)`. The secret **lattice point** is `v = 2*b1 + (-1)*b2` (computed with the good basis).
2. Add a small random **error vector** `e` (this is the "noise" that hides the point).
3. The **ciphertext** is the target point `t = v + e`, which is a point sitting *near* a lattice point but not on it.

Decrypting means: given `t`, find the nearest lattice point (that's the **Closest Vector Problem**) to recover `v`, hence the message.

In [ ]:
np.random.seed(0)
msg = np.array([2, -1])                 # the secret message = lattice coordinates

# The true lattice point that encodes the message (uses the good/secret basis)
v = # COMPLETE THIS CODE

# Small error vector to hide it
error = np.array([0.35, -0.30])
t = # COMPLETE THIS CODE           # ciphertext = target point

print('message coordinates :', msg.tolist())
print('secret lattice point v =', v.tolist())
print('error vector e =', error.tolist(), ' (length', round(np.linalg.norm(error), 3), ')')
print('CIPHERTEXT  t = v + e =', np.round(t, 3).tolist())

#### **Problem #2.5**

**Decrypt with the good (secret) basis.** Run Babai's rounding algorithm using `B_good`: write `t` in good-basis coordinates, round to the nearest integers, and map back to a lattice point. Because the good basis is short and near-orthogonal, the rounding lands on the correct point `v`.

In [ ]:
recovered_point, recovered_coords = babai_round(# COMPLETE THIS CODE

print('decrypted lattice point :', recovered_point.tolist())
print('decrypted coordinates   :', recovered_coords.tolist())
print('true message            :', msg.tolist())
print('SUCCESS!' if # COMPLETE THIS CODE else 'FAILED')

#### **Problem #2.6**

**Now try to decrypt with the bad basis.** Run the exact same Babai rounding, but pass `B_bad` instead. The bad basis has long, skewed cells, so rounding in its coordinate system lands on the **wrong** lattice point — decryption fails. This is what an attacker (who only has the public bad basis) is stuck with.

In [ ]:
bad_point, bad_coords = babai_round(# COMPLETE THIS CODE

print('bad-basis decrypted point :', bad_point.tolist())
print('true lattice point v      :', v.tolist())
print('correct?', np.allclose(bad_point, v))
print('distance from true point  :', round(np.linalg.norm(bad_point - v), 3))

#### **Reflection**

* Both bases describe the *same* lattice, and we ran the *same* rounding algorithm. So why does the good basis decrypt correctly while the bad basis fails?

* How is this the lattice version of a "trapdoor"? Which basis is the public key, and which is the private key?

# End of Notebook

---
© 2026 Frontier Technology Institute, All rights reserved